# CSA — full Karpathy TRAIN: first10 XE epochs + validation
**GPU T4×2, Internet ON, Save Version → Save & Run All.** No API Secret or old checkpoint.
Add **csa-full-roi-karpathy-v134** and **csa-semantic-banks-v130**; edit only their paths below if needed.

Fresh self-written CSA adaptation, ordinary2GPU FP32/global32captions.113287TRAIN images/566747references per epoch,
17711updates/epoch;10epochs177110updates. Warmup3full epochs8e-6→1e-4, thenflat; no SCST.
After each epoch: all5000val images, beam5/max20/rawscore + CIDEr and NLL; val never joins training, test unused.
Frozen CLIP inputs already cached; learned CSA representations recomputed each update.

Train-only estimate≈5.18h; validation/checkpoints/preparation add time, mature decoding not yet timed.
Wall cap10.5h is a stop limit. Mandatory six DDP guards + original per-image beam contract must pass before training.
V2 uses the established `model.beam_search` for validation; the new batched decoder was removed after its ID comparison failed.
The failed first attempt stopped BEFORE training. V2 uses a separate output folder and needs no resume checkpoint.
Model execution is performed by you on Kaggle; local agent checks were static/NumPy only.

`checkpoint.bin` stores latest model/Adam/cursor/both-rank RNG every1000updates and before/after validation.
`best.bin` stores greatest valid CIDEr (ties earlier). **Preserve entire output folder**, not just the small review ZIP.
To resume, mount that complete folder and set `RESUME_ROOT`; use a fresh session/output path and this same pinned notebook.
Do not point it at checkpoint131/QFormer/A. Extension beyond10epochs needs a separate approved schedule.
Send `csa_full_xe_10_v2_review.zip`. Hallucination/CHAIR and test metrics are not computed here.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
# Leave empty for a fresh run. For continuation mount the ENTIRE previous output folder.
RESUME_ROOT=''
OUTPUT=Path('/kaggle/working/csa_full_xe_10_v2')
WALL_HOURS=10.5  # Stop limit, not an ETA or a promise of available quota.
COMMIT='e655a565d8be354abe579b8a2bb59e7d1733803b'
HASHES={'run_sharing_benchmark.py': '51706ae5cbd96fb0484cb0487a05f14f1b34f4332bd9cb57a299d7d3d5a14d18', 'sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'pilot_data.py': '0edab33c7ad7cc54c28efdf6007f23fa6dc4547bdda32e5c0d7d888a3369394d', 'package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'DDP_SHARING_BENCHMARK_PROTOCOL.md': '6537e5e44fb8eb1504c4234b87d4e4e87929cf80fed3884f3caa1ef09468b2f1', 'run_full_xe.py': 'ba0593c8245a8b7bc8b4b82818972d433890a04eb563cdfaf31d7e1ebbb33370', 'full_xe_data.py': 'fb6ca64e74e1799c9fcdf014b5e9137647ca060b93516f02ce1c6fe634f9f8a6', 'full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'test_full_xe.py': 'aee46db054fd18bcf9306d8d8240414d67ca39dccb74d7ab0337e5d9ef7cb1fa', 'FULL_XE_10_PROTOCOL.md': 'b52d4081cf0657c7c7fba8a00def7efbff928b4979180e08b4e552cc3983a70c'}
if not FULL_ROOT.is_dir() or not BANK_ROOT.is_dir():raise FileNotFoundError('Add BOTH fullROI134 and semantic-bank130 Datasets.')
if RESUME_ROOT and not Path(RESUME_ROOT).is_dir():raise FileNotFoundError('RESUME_ROOT must be the mounted complete previous output folder.')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; start a fresh session/output path. No automatic overwrite.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for stage/checkpoint/temp files.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-full-xe10-'));STAGE=SOURCE/'staged-full-cache';LOG=OUTPUT/'notebook.log'
os.environ['OMP_NUM_THREADS']='1'
for n,h in HASHES.items():
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(data).hexdigest()!=h:raise ValueError('Changed source:'+n)
    p=SOURCE/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(data)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')
print('Full10 XE source:',COMMIT,flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Full XE active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Full XE wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False;parent_deadline=time.monotonic()+WALL_HOURS*3600
common=['--bank-root',BANK_ROOT,'--stage',STAGE,'--output',OUTPUT]
try:
    run([sys.executable,'-u',SOURCE/'test_full_xe.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2'],timeout=300)
    run([sys.executable,'-u',SOURCE/'run_full_xe.py','prepare','--full-root',FULL_ROOT,*common],timeout=900)
    run([sys.executable,'-u',SOURCE/'run_full_xe.py','metric_preflight',*common],timeout=300)
    preflight=OUTPUT/'preflight';preflight.mkdir()
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'run_sharing_benchmark.py','worker','--bank-root',BANK_ROOT,'--stage',STAGE,'--output',preflight],timeout=600)
    if json.loads((preflight/'result.json').read_text(encoding='utf-8'))['preflight_cases']!=6:raise RuntimeError('Missing DDP preflight')
    remaining=parent_deadline-time.monotonic()
    if remaining<600:raise TimeoutError('No training budget after preparation')
    run([*launch,SOURCE/'run_full_xe.py','worker',*common,'--resume-root',RESUME_ROOT,'--wall-seconds',remaining-60],timeout=remaining)
    result=OUTPUT/'result.json'
    complete=result.is_file() and json.loads(result.read_text(encoding='utf-8')).get('status')=='FULL_XE_10_COMPLETE'
    if not complete:raise RuntimeError('Stopped before10epochs; preserve ENTIRE output to resume. Do not report full completion.')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,requested_epochs=10,test_used=False)),encoding='utf-8')
    review=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True)
    print('KEEP ENTIRE FOLDER:',OUTPUT,flush=True)
    for name in ('checkpoint.bin','best.bin'):
        if (OUTPUT/name).is_file():print('KEEP CHECKPOINT:',OUTPUT/name,flush=True)
    if not (OUTPUT/'checkpoint.bin').is_file():print('No training checkpoint exists yet; do not set RESUME_ROOT to this output.',flush=True)
    print('COMPLETE10' if complete else 'INCOMPLETE; preserve output and resume, no automatic restart.',flush=True)
